# Diccionario de articulos — texto completo por norma

Construye `data/diccionario_articulos.csv` (fuente, numero, texto_completo) a partir de los
articulos unicos citados en `dataset_cross_encoder.csv` (ver `data/articulos_a_buscar.csv`).

Ver `data/README.md` para el diseno completo (por que es un archivo separado del dataset de
pares, y como se usa en el entrenamiento/inferencia).

**Fuentes** (`data/fuentes_normas.csv`, resueltas manualmente via WebSearch — cubren la
inmensa mayoria de los articulos):
- CST y Constitucion Politica: pagina unica por norma en `alcaldiabogota.gov.co` (Regimen
  Legal — Secretaria Juridica Distrital, en realidad un repositorio nacional, no solo distrital).
  `secretariasenado.gov.co` y `suin-juriscol.gov.co`, las fuentes obvias, estan caidas —
  verificado tanto desde este entorno como desde fuera (WebFetch), no es un problema local.
- Leyes/Decretos/Resoluciones: mayoria en `funcionpublica.gov.co/eva/gestornormativo` (su
  buscador es una app JS, no se puede armar la URL por patron — cada norma se resolvio una
  vez a mano via WebSearch, no cambia con el tiempo). Nota: el sitio tiene un problema de
  cadena de certificado SSL (confirmado con `openssl s_client`: el certificado en si es
  valido, vigente y de una CA real — Sectigo — pero el servidor no manda la cadena completa),
  asi que este notebook usa `verify=False` solo para ese dominio.
- 2 articulos sin fuente publica: una "Convención Colectiva de Trabajo con Sintracerromatoso"
  y una CCT sin identificar de otra sentencia — son instrumentos privados entre una empresa y
  un sindicato especifico, no estan codificados publicamente. Quedan fuera del diccionario.

**De paso, esta resolucion destapo varias citas mal etiquetadas en el dataset** (ya corregidas
directamente en `dataset_cross_encoder.csv`): "Decreto 2177 de 1998" → 1989, "Decreto 2400 de
1979" → en realidad es una Resolucion no un Decreto, "Decreto 2463 de 2011" → 2001 (no existe
un decreto con ese numero en 2011, solo el typo del extractor).

## 1. Setup

In [1]:
# En Colab: descomenta la siguiente linea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q beautifulsoup4 requests pandas

import os
import re
import html
import time

import requests
import urllib3
import pandas as pd
from bs4 import BeautifulSoup

# funcionpublica.gov.co no manda la cadena de certificado SSL completa (ver nota de la celda
# de arriba) -- este notebook usa verify=False a propósito solo para ese dominio, así que se
# apaga el warning correspondiente en vez de dejarlo repetirse en cada request.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {"User-Agent": "Mozilla/5.0"}

DATASET_PATH = "../../data/dataset_cross_encoder.csv"
FUENTES_PATH = "../../data/fuentes_normas.csv"
OBJETIVO_PATH = "../../data/articulos_a_buscar.csv"
OUTPUT_PATH = "../../data/diccionario_articulos.csv"

df_fuentes = pd.read_csv(FUENTES_PATH)

print(f"Fuentes a descargar: {len(df_fuentes)}")

Fuentes a descargar: 41


## 1b. Regenerar `articulos_a_buscar.csv` desde el dataset

`data/articulos_a_buscar.csv` es 100% derivable de la columna `articulo` de
`dataset_cross_encoder.csv` — se regenera aqui en vez de depender de un archivo suelto, para
que este notebook no dependa de un script externo que no queda guardado en ningun lado.

Normaliza cada cita (`"CST Art. 127"`, `"Ley 50 de 1990, Art. 15"`,
`"Artículo 62 del Código Sustantivo del Trabajo"`, etc.) a un par `(fuente, numero)`: unifica
alias de fuente (`CST` = `Código Sustantivo del Trabajo`, `CP` = `Constitución Política`),
separa citas de multiples articulos (`"Arts. 11, 16, 17 y 27"`), e ignora sufijos de
numeral/literal/paragrafo (se agrupa a nivel de articulo completo, no de sub-numeral).

In [2]:
ALIAS_FUENTE = {
    "cst": "CST", "código sustantivo del trabajo": "CST",
    "cp": "Constitución Política", "constitución política": "Constitución Política",
    "cgp": "CGP", "cptss": "CPTSS",
}

def canonicalizar_fuente(f):
    return ALIAS_FUENTE.get(f.strip().lower().rstrip(","), f.strip())

def normalizar_articulos_citados(df_pares):
    registros = []
    for a in df_pares["articulo"].unique():
        a = a.strip()

        # caso especial: "Articulo 57 numeral 5 del Codigo Sustantivo del Trabajo" no calza
        # con el patron general por el "numeral 5" intercalado antes del "del"
        if a == "Artículo 57 numeral 5 del Código Sustantivo del Trabajo":
            registros.append({"fuente": "CST", "numero": "57"})
            continue

        # patron "Articulo N de/del <FUENTE>..."
        m = re.match(r"^Art[íi]culo\s+(\d+[A-Za-z\-]*)\s+(?:de\s+la\s+|del\s+|de\s+)(.+)$", a, re.IGNORECASE)
        if m:
            numero, fuente_raw = m.groups()
            fuente_raw = re.split(r"\s+sobre\s+", fuente_raw)[0]  # corta "... sobre pago de cesantías"
            registros.append({"fuente": canonicalizar_fuente(fuente_raw), "numero": numero.rstrip("-")})
            continue

        # patron "<FUENTE> [,] Art[iculo/s]. N[, N, ...]"
        m = re.match(r"^(.+?),?\s+Art[íi]?c?u?l?o?s?\.?\s+(\d[\dA-Za-z,\sy\-]*)", a, re.IGNORECASE)
        if m:
            fuente_raw, numeros_raw = m.groups()
            fuente = canonicalizar_fuente(fuente_raw)
            if re.search(r",\s*\d+.*\sy\s+\d+", numeros_raw) or (" y " in numeros_raw and re.match(r"^\d", numeros_raw.strip())):
                # cita multiples articulos: "Arts. 11, 16, 17 y 27"
                for n in re.findall(r"\d+[A-Za-z\-]*", numeros_raw):
                    registros.append({"fuente": fuente, "numero": n.rstrip("-")})
            else:
                pn = re.match(r"\d+[A-Za-z\-]*", numeros_raw.strip())
                numero = (pn.group(0) if pn else numeros_raw.strip()).rstrip("-")
                registros.append({"fuente": fuente, "numero": numero})
            continue

        # sin numero de articulo especifico (cita una ley/decreto/resolucion completa, ej.
        # "Ley 361 de 1997") -- no entra al diccionario por articulo, se deja fuera a proposito

    df_reg = pd.DataFrame(registros)
    return (df_reg.groupby(["fuente", "numero"], as_index=False).size()
            .rename(columns={"size": "n_citas"}).sort_values(["fuente", "numero"]))

df_pares = pd.read_csv(DATASET_PATH)
df_objetivo = normalizar_articulos_citados(df_pares)
df_objetivo.to_csv(OBJETIVO_PATH, index=False)
df_objetivo["numero"] = df_objetivo["numero"].astype(str)

print(f"Citas originales unicas en el dataset: {df_pares['articulo'].nunique()}")
print(f"articulos_a_buscar.csv regenerado: {len(df_objetivo)} articulos unicos (fuente+numero)")

Citas originales unicas en el dataset: 228
articulos_a_buscar.csv regenerado: 145 articulos unicos (fuente+numero)


## 2. Descarga y extraccion de articulos por fuente

Cada fuente es una sola pagina con la norma completa (o casi completa). Se descarga una vez,
se separa en articulos individuales por el patron `ARTICULO N.` / `ARTÍCULO N°.` (verificado
contra ejemplos reales de ambos dominios antes de correr el lote completo — CST en
alcaldiabogota.gov.co y Ley 789/2002 en funcionpublica.gov.co usan el mismo patron), y se
guarda un diccionario `{numero: texto}` por fuente.

In [3]:
RE_ARTICULO = re.compile(
    r'ART[IÍ]CULO\s+(\d+[A-Za-z]?)\s*[o°º]?\s*(?:\.|-|(?=\s+[A-ZÁÉÍÓÚÑ]))',
    re.IGNORECASE,
)
# v3: ademas de punto, admite guion pegado al numero ("ARTICULO 6- Causales...") o directo a
# la siguiente palabra en mayuscula sin puntuacion ("Articulo 11 Entiendese...") -- el guion y
# el lookahead de mayuscula se agregaron tras revisar casos reales que fallaban. Sigue
# exigiendo la palabra completa ARTICULO/Articulo (nunca "art." abreviado), que es lo que evita
# que listas de referencias tipo "art. 6, tema, art. 7, tema..." se confundan con encabezados
# reales de articulo.

def descargar_html(url, dominio):
    # funcionpublica.gov.co tiene un problema de cadena de certificado SSL (el certificado
    # en si es valido y vigente, ver nota en la celda de arriba) -- se usa verify=False solo
    # para ese dominio, verificado manualmente antes de automatizar.
    verify = dominio != "funcionpublica"
    resp = requests.get(url, headers=HEADERS, timeout=30, verify=verify)
    resp.raise_for_status()
    return resp.content

def extraer_texto_plano(contenido_bytes, dominio):
    # alcaldiabogota.gov.co declara windows-1252 explicitamente en su meta tag (correcto).
    # funcionpublica.gov.co manda "charset=UTF-8" en el header HTTP pero el meta tag interno
    # dice "ISO-8859-1" -- BeautifulSoup confia en el meta tag equivocado y corrompe el texto
    # (mojibake tipo "FunciÃ³n"), verificado al revisar un caso con 0 articulos extraidos pese
    # a tener texto de sobra -- forzamos utf-8 para ese dominio, que es lo que el header
    # (correcto) dice.
    if dominio == "alcaldiabogota":
        encoding = "windows-1252"
    elif dominio == "funcionpublica":
        encoding = "utf-8"
    else:
        encoding = None
    soup = BeautifulSoup(contenido_bytes, "html.parser", from_encoding=encoding)
    texto = soup.get_text(" ")
    texto = html.unescape(texto)
    texto = re.sub(r"[\s\xa0]+", " ", texto).strip()
    return texto

def extraer_articulos(texto_plano):
    posiciones = [(m.start(), m.group(1)) for m in RE_ARTICULO.finditer(texto_plano)]
    articulos = {}
    for i, (pos, numero) in enumerate(posiciones):
        fin = posiciones[i + 1][0] if i + 1 < len(posiciones) else len(texto_plano)
        texto_articulo = texto_plano[pos:fin].strip()
        # ordinal "o" (ej. "2o", "4o") es marcador de orden, no letra de sub-articulo real
        # (esas usan A/B, ej. "239A") -- se guarda tambien la version sin el "o" final para
        # que calce con el numero "plano" que usa articulos_a_buscar.csv
        claves = {numero}
        if re.fullmatch(r"\d+o", numero, re.IGNORECASE):
            claves.add(numero[:-1])
        for clave in claves:
            if clave not in articulos or len(texto_articulo) > len(articulos[clave]):
                articulos[clave] = texto_articulo
    return articulos

print("Funciones listas.")

Funciones listas.


In [4]:
articulos_por_fuente = {}
fuentes_fallidas = []

for i, row in df_fuentes.iterrows():
    fuente, url, dominio = row["fuente"], row["url"], row["dominio"]
    print(f"[{i+1}/{len(df_fuentes)}] {fuente} ({dominio})...", end=" ")
    try:
        contenido = descargar_html(url, dominio)
        texto = extraer_texto_plano(contenido, dominio)
        articulos = extraer_articulos(texto)
        articulos_por_fuente[fuente] = articulos
        print(f"{len(articulos)} articulos extraidos ({len(texto)} chars totales)")
    except Exception as e:
        fuentes_fallidas.append({"fuente": fuente, "error": str(e)})
        print(f"ERROR: {e}")
    time.sleep(0.5)

print(f"\nFuentes procesadas: {len(articulos_por_fuente)}/{len(df_fuentes)}")
if fuentes_fallidas:
    print("Fallidas:")
    for f in fuentes_fallidas:
        print(" ", f)

[1/41] CST (alcaldiabogota)... 

507 articulos extraidos (495735 chars totales)


[2/41] Constitución Política (alcaldiabogota)... 

382 articulos extraidos (585951 chars totales)


[3/41] Decreto 1045 de 1978 (funcionpublica)... 

...

[40/41] Ley 30 de 1992 (funcionpublica)... 

158 articulos extraidos (96104 chars totales)


[41/41] Ley 1595 de 2012 (funcionpublica)... 

18 articulos extraidos (32130 chars totales)



Fuentes procesadas: 41/41


## 3. Cruce contra los articulos objetivo

In [5]:
filas = []
faltantes = []

for _, row in df_objetivo.iterrows():
    fuente, numero, n_citas = row["fuente"], row["numero"], row["n_citas"]
    articulos = articulos_por_fuente.get(fuente, {})
    texto = articulos.get(numero)
    if texto is None:
        faltantes.append({"fuente": fuente, "numero": numero, "n_citas": n_citas,
                           "razon": "fuente_sin_mapeo" if fuente not in articulos_por_fuente else "articulo_no_encontrado_en_fuente"})
        continue
    url_fuente = df_fuentes.loc[df_fuentes["fuente"] == fuente, "url"].values[0]
    filas.append({"fuente": fuente, "numero": numero, "texto_completo": texto,
                   "n_citas_en_dataset": n_citas, "url_fuente": url_fuente})

df_diccionario = pd.DataFrame(filas)
df_diccionario.to_csv(OUTPUT_PATH, index=False)

print(f"Diccionario guardado en {OUTPUT_PATH}: {len(df_diccionario)}/{len(df_objetivo)} articulos")
print(f"Cobertura: {len(df_diccionario) / len(df_objetivo):.1%}")

if faltantes:
    print(f"\n--- {len(faltantes)} articulos objetivo SIN texto (revisar) ---")
    df_faltantes = pd.DataFrame(faltantes).sort_values(["razon", "fuente"])
    print(df_faltantes.to_string(index=False))
    df_faltantes.to_csv("../../data/diccionario_faltantes.csv", index=False)

Diccionario guardado en ../../data/diccionario_articulos.csv: 143/145 articulos
Cobertura: 98.6%

--- 2 articulos objetivo SIN texto (revisar) ---
                                               fuente numero  n_citas            razon
                                                  CCT     40        1 fuente_sin_mapeo
Convención Colectiva de Trabajo con Sintracerromatoso     14        1 fuente_sin_mapeo


### 3b. Correcciones manuales — articulos "sustitutivos" que el separador corta mal

3 articulos son del tipo "El articulo N de la Ley X quedara asi: ARTICULO N. [texto nuevo]"
(una ley que sustituye el texto de un articulo de OTRA ley, citando el nuevo texto completo
adentro). El separador automatico (seccion 2) trata ese "ARTICULO N" anidado como si fuera un
limite real, y corta el articulo original a solo unas palabras. Verificado a mano contra la
fuente para los 3 casos que el chequeo de calidad (seccion 4) marco como sospechosamente
cortos — sin esto, quedarian con 15-16 caracteres de texto en vez del articulo completo.

In [6]:
SOBREESCRITURAS_MANUALES = {
    ("Ley 1232 de 2008", "1"): """ARTÍCULO 1º. El Artículo 2º de la Ley 82 de 1993 quedará así: ARTÍCULO 2º. Jefatura femenina de hogar. Para los efectos de la presente ley, la Jefatura Femenina de Hogar, es una categoría social de los hogares, derivada de los cambios sociodemográficos, económicos, culturales y de las relaciones de género que se han producido en la estructura familiar, en las subjetividades, representaciones e identidades de las mujeres que redefinen su posición y condición en los procesos de reproducción y producción social, que es objeto de políticas públicas en las que participan instituciones estatales, privadas y sectores de la sociedad civil. En concordancia con lo anterior, es Mujer Cabeza de Familia, quien, siendo soltera o casada, ejerce la jefatura femenina de hogar y tiene bajo su cargo, afectiva, económica o socialmente, en forma permanente, hijos menores propios u otras personas incapaces o incapacitadas para trabajar, ya sea por ausencia permanente o incapacidad física, sensorial, síquica o moral del cónyuge o compañero permanente o deficiencia sustancial de ayuda de los demás miembros del núcleo familiar. PARÁGRAFO. La condición de Mujer Cabeza de Familia y la cesación de la misma, desde el momento en que ocurra el respectivo evento, deberá ser declarada ante notario por cada una de ellas, expresando las circunstancias básicas del respectivo caso y sin que por este concepto se causen emolumentos notariales a su cargo.""",
    ("Ley 1822 de 2017", "2"): """ARTÍCULO 2°. El artículo 239 del Código Sustantivo del Trabajo, quedará así: "ARTÍCULO 239. Prohibición de despido. 1. Ninguna trabajadora podrá ser despedida por motivo dé embarazo o lactancia sin la autorización previa del Ministerio de Trabajo que avale una justa causa. 2. Se presume el despido efectuado por motivo de embarazo o lactancia, cuando este haya tenido lugar dentro del período de embarazo y/o dentro de los tres meses posteriores al parto. 3. Las trabajadoras que trata el numeral uno (1) de este artículo, que sean despedidas sin autorización de las autoridades competentes, tendrán derecho al pago adicional de una indemnización igual a sesenta días (60) días de trabajo, fuera de las indemnizaciones y prestaciones a que hubiere lugar de acuerdo con su contrato de trabajo. 4. En el caso de la mujer trabajadora que por alguna razón excepcional no disfrute de la semana preparto obligatoria, y/o de algunas de las diecisiete (17) semanas de descanso, tendrá derecho al pago de las semanas que no gozó de licencia. En caso de parto múltiple tendrá el derecho al pago de dos (2) semanas adicionales y, en caso de que el hijo sea prematuro, al pago de la diferencia de tiempo entre la fecha del alumbramiento y el nacimiento a término".""",
    ("Ley 50 de 1990", "35"): """ARTÍCULO 35. El artículo 239 del Código Sustantivo del Trabajo quedará así: Artículo 239. Prohibición de despedir. 1. Ninguna trabajadora puede ser despedida por motivo de embarazo o lactancia. 2. Se presume que el despido se ha efectuado por motivo de embarazo o lactancia, cuando ha tenido lugar dentro del período de embarazo o dentro de los tres meses posteriores al parto, y sin autorización de las autoridades de que trata el artículo siguiente. 3. La trabajadora despedida sin autorización de la autoridad tiene derecho al pago de una indemnización equivalente a los salarios de sesenta (60) días, fuera de las indemnizaciones y prestaciones a que hubiera lugar de acuerdo con el contrato de trabajo y, además, al pago de las doce (12) semanas de descanso remunerado de que trata este capítulo, si no lo ha tomado.""",
}

n_corregidos = 0
for (fuente, numero), texto in SOBREESCRITURAS_MANUALES.items():
    mask = (df_diccionario["fuente"] == fuente) & (df_diccionario["numero"].astype(str) == numero)
    if mask.sum() == 1:
        df_diccionario.loc[mask, "texto_completo"] = texto
        n_corregidos += 1

df_diccionario.to_csv(OUTPUT_PATH, index=False)
print(f"{n_corregidos}/{len(SOBREESCRITURAS_MANUALES)} correcciones manuales aplicadas y guardadas.")

3/3 correcciones manuales aplicadas y guardadas.


## 4. Vista rapida de calidad

Longitud de texto por articulo — un articulo sospechosamente corto (ej. < 30 caracteres)
probablemente se corto mal en el limite con el siguiente articulo.

In [7]:
df_diccionario["len_texto"] = df_diccionario["texto_completo"].str.len()
print(df_diccionario["len_texto"].describe())
print()
sospechosos = df_diccionario[df_diccionario["len_texto"] < 30]
print(f"Articulos sospechosamente cortos (<30 chars): {len(sospechosos)}")
if len(sospechosos):
    print(sospechosos[["fuente", "numero", "texto_completo"]].to_string(index=False))
df_diccionario.sample(min(5, len(df_diccionario)))[["fuente", "numero", "texto_completo"]]

count     143.000000
mean     1294.881119
std      1430.267549
min        56.000000
25%       440.500000
50%       851.000000
75%      1475.000000
max      8706.000000
Name: len_texto, dtype: float64

Articulos sospechosamente cortos (<30 chars): 0


,fuente,numero,texto_completo
28,CST,241,"ARTICULO 241. Modificado por el art. 8, Decret..."
85,Decreto 2351 de 1965,7,ARTÍCULO 7. Terminación del contrato por justa...
106,Ley 115 de 1994,115,ARTÍCULO 115.- Régimen especial de los educado...
43,CST,58,ARTICULO 58. OBLIGACIONES ESPECIALES DEL TRABA...
83,Decreto 2351 de 1965,3,ARTÍCULO 3 . Contratistas independientes. 1. S...
